In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import time
import random
import re
from tqdm import tqdm
import os

In [2]:
# Config
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Accept-Language': 'vi-VN,vi;q=0.9,en-US;q=0.8,en;q=0.7',
}

BASE_URL = 'https://www.dienmayxanh.com'
MIN_DELAY = 0.5
MAX_DELAY = 1.5

## 1. Lấy danh sách Categories

In [8]:
def get_dmx_categories():
    categories = [
        {'name': "Món canh", 'url': 'https://www.dienmayxanh.com/vao-bep/mon-canh'},
        {'name': "Món bánh", 'url': 'https://www.dienmayxanh.com/vao-bep/mon-banh'},
        {'name': "Món chè", 'url': 'https://www.dienmayxanh.com/vao-bep/mon-che'},
        {'name': "Món nước", 'url': 'https://www.dienmayxanh.com/vao-bep/mon-nuoc'},
        {'name': "Món kem", 'url': 'https://www.dienmayxanh.com/vao-bep/mon-kem'},
        {'name': "Món nướng", 'url': 'https://www.dienmayxanh.com/vao-bep/mon-nuong'},
        {'name': "Món cháo", 'url': 'https://www.dienmayxanh.com/vao-bep/mon-chao'},
        {'name': "Món xào", 'url': 'https://www.dienmayxanh.com/vao-bep/mon-xao'},

        {'name': "Món kho", 'url': 'https://www.dienmayxanh.com/vao-bep/mon-kho'},
        {'name': "Món chiên", 'url': 'https://www.dienmayxanh.com/vao-bep/mon-chien'},        
        {'name': "Món hấp", 'url': 'https://www.dienmayxanh.com/vao-bep/mon-hap'},
        {'name': "Món lẩu", 'url': 'https://www.dienmayxanh.com/vao-bep/mon-lau'},
        {'name': "Món gỏi - salad", 'url': 'https://www.dienmayxanh.com/vao-bep/mon-goi-tron'},
        {'name': "Món từ gà", 'url': 'https://www.dienmayxanh.com/vao-bep/mon-tu-ga'},
        {'name': "Món từ bò", 'url': 'https://www.dienmayxanh.com/vao-bep/mon-tu-bo'},
        {'name': "Món chay", 'url': 'https://www.dienmayxanh.com/vao-bep/mon-chay'},
        {'name': "Ăn vặt", 'url': 'https://www.dienmayxanh.com/vao-bep/an-vat'},

        {'name': "Ngày lễ Tết", 'url': 'https://www.dienmayxanh.com/vao-bep/ngay-le-tet'},
        {'name': "Thức uống", 'url': 'https://www.dienmayxanh.com/vao-bep/thuc-uong'},
        {'name': "Sinh tố", 'url': 'https://www.dienmayxanh.com/vao-bep/sinh-to'},
        {'name': "Trà sữa", 'url': 'https://www.dienmayxanh.com/vao-bep/tra-sua'},
        {'name': "Nước ép", 'url': 'https://www.dienmayxanh.com/vao-bep/nuoc-ep'},
        {'name': "Món tráng miệng", 'url': 'https://www.dienmayxanh.com/vao-bep/mon-trang-mieng'},
        {'name': "Món khô - mắm", 'url': 'https://www.dienmayxanh.com/vao-bep/mon-kho-mam'},
        {'name': "Món cuốn - trộn", 'url': 'https://www.dienmayxanh.com/vao-bep/mon-cuon-tron'}
    ]
    return categories

categories = get_dmx_categories()
print(f"Tổng số category: {len(categories)}")

Tổng số category: 25


In [15]:
categories

[{'name': 'Món canh', 'url': 'https://www.dienmayxanh.com/vao-bep/mon-canh'},
 {'name': 'Món bánh', 'url': 'https://www.dienmayxanh.com/vao-bep/mon-banh'},
 {'name': 'Món chè', 'url': 'https://www.dienmayxanh.com/vao-bep/mon-che'},
 {'name': 'Món nước', 'url': 'https://www.dienmayxanh.com/vao-bep/mon-nuoc'},
 {'name': 'Món kem', 'url': 'https://www.dienmayxanh.com/vao-bep/mon-kem'},
 {'name': 'Món nướng', 'url': 'https://www.dienmayxanh.com/vao-bep/mon-nuong'},
 {'name': 'Món cháo', 'url': 'https://www.dienmayxanh.com/vao-bep/mon-chao'},
 {'name': 'Món xào', 'url': 'https://www.dienmayxanh.com/vao-bep/mon-xao'},
 {'name': 'Món kho', 'url': 'https://www.dienmayxanh.com/vao-bep/mon-kho'},
 {'name': 'Món chiên', 'url': 'https://www.dienmayxanh.com/vao-bep/mon-chien'},
 {'name': 'Món hấp', 'url': 'https://www.dienmayxanh.com/vao-bep/mon-hap'},
 {'name': 'Món lẩu', 'url': 'https://www.dienmayxanh.com/vao-bep/mon-lau'},
 {'name': 'Món gỏi - salad',
  'url': 'https://www.dienmayxanh.com/vao-b

## 2. Lấy danh sách URL công thức

Do trang điện máy xanh có định dạng cần bấm nút "Xem thêm" thay vì duyệt qua các page nên ta cần dùng selenium để giả lập click nút xem thêm này

In [14]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, ElementClickInterceptedException
import time

def get_food_urls_from_category_dmx(category_url):
    driver = webdriver.Chrome()
    driver.get(category_url)
    wait = WebDriverWait(driver, 10)

    food_urls = set()   # tránh trùng lặp

    prev_count = 0

    while True:
        # 1. Try to click "Xem thêm"
        try:
            see_more_btn = wait.until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, "a.seemore-cook"))
            )
            driver.execute_script("arguments[0].scrollIntoView(true);", see_more_btn)
            time.sleep(0.3)
            see_more_btn.click()
        except TimeoutException:
            # Không còn nút => load hết rồi
            break
        except ElementClickInterceptedException:
            # Thử lại nhẹ
            time.sleep(1)
            continue

        # 2. Wait until new li are loaded
        time.sleep(1)  # nhẹ để JS load
        all_items = driver.find_elements(By.CSS_SELECTOR, "ul li a")
        cur_count = len(all_items)

        if cur_count == prev_count:
            # Không tăng nữa => hết dữ liệu
            break

        prev_count = cur_count

    # 3. Extract URL từ tất cả li
    final_items = driver.find_elements(By.CSS_SELECTOR, "ul li a")
    for item in final_items:
        href = item.get_attribute("href")
        if href and "/vao-bep/" in href:   # filter chuẩn
            food_urls.add(href)

    driver.quit()
    return list(food_urls)

In [19]:
# Test trên 1 category
print(f"Testing: {categories[0]['name']}")

test_urls = get_food_urls_from_category_dmx(categories[0]['url'])
print(f"Found {len(test_urls)} recipes")

Testing: Món canh
Found 960 recipes


In [20]:
test_urls[:3]

['https://www.dienmayxanh.com/vao-bep/2-cach-nau-canh-rau-ngot-chay-dau-hu-thom-ngon-thanh-mat-don-11520',
 'https://www.dienmayxanh.com/vao-bep/cach-nau-ga-ac-tan-ham-ngai-cuu-cuc-ngon-bo-mau-huyet-01384',
 'https://www.dienmayxanh.com/vao-bep/3-cach-nau-canh-dua-leo-xuong-heo-nhoi-thit-va-tom-thanh-mat-05075']

In [21]:
# Cào tất cả categories
all_recipe_urls = []
seen_urls = set()

for category in tqdm(categories, desc="Crawling categories"):
    cat_name = category["name"]
    cat_url  = category["url"]

    print(f"Crawling category: {cat_name}")

    try:
        food_urls = get_food_urls_from_category_dmx(cat_url)
    except Exception as e:
        print(f"Lỗi khi crawl category {cat_name}: {e}")
        continue

    for url in food_urls:
        if url not in seen_urls:
            seen_urls.add(url)
            all_recipe_urls.append((cat_name, url))

    print(f"Total collected: {len(all_recipe_urls)}")
    time.sleep(random.uniform(1, 2))

print(f"Total unique recipes: {len(all_recipe_urls)}")


Crawling categories:   0%|          | 0/25 [00:00<?, ?it/s]

Crawling category: Món canh
Total collected: 960


Crawling categories:   4%|▍         | 1/25 [01:30<36:10, 90.45s/it]

Crawling category: Món bánh
Total collected: 3518


Crawling categories:   8%|▊         | 2/25 [08:08<1:44:07, 271.63s/it]

...

Crawling category: Món khô - mắm
Total collected: 13394


Crawling categories:  96%|█████████▌| 24/25 [33:46<00:55, 55.51s/it]

Crawling category: Món cuốn - trộn
Total collected: 13515


Crawling categories: 100%|██████████| 25/25 [34:34<00:00, 82.99s/it]

Total unique recipes: 13515


In [23]:
all_food_urls = [(cat, url) for cat, url in all_recipe_urls]


In [24]:
all_food_urls

[('Món canh',
  'https://www.dienmayxanh.com/vao-bep/2-cach-nau-canh-rau-ngot-chay-dau-hu-thom-ngon-thanh-mat-don-11520'),
 ('Món canh',
  'https://www.dienmayxanh.com/vao-bep/cach-nau-ga-ac-tan-ham-ngai-cuu-cuc-ngon-bo-mau-huyet-01384'),
 ('Món canh',
  'https://www.dienmayxanh.com/vao-bep/3-cach-nau-canh-dua-leo-xuong-heo-nhoi-thit-va-tom-thanh-mat-05075'),
 ('Món canh',
  'https://www.dienmayxanh.com/vao-bep/cach-nau-sup-nam-can-tay-de-lam-de-an-thom-nuc-mui-07891'),
 ('Món canh',
  'https://www.dienmayxanh.com/vao-bep/cach-nau-sup-cua-chay-thom-ngon-don-gian-02693'),
 ('Món canh',
  'https://www.dienmayxanh.com/vao-bep/cac-mon-canh-mua-dong-mien-bac-de-nau-giu-am-co-the-23616'),
 ('Món canh',
  'https://www.dienmayxanh.com/vao-bep/cach-nau-gio-heo-ham-cu-cai-muoi-thom-ngon-bo-duong-ca-nha-07674'),
 ('Món canh',
  'https://www.dienmayxanh.com/vao-bep/cach-nau-canh-ca-minh-thai-han-quoc-bo-duong-la-mieng-cuc-09540'),
 ('Món canh',
  'https://www.dienmayxanh.com/vao-bep/cach-lam-ca-me

In [25]:
all_foods_df = pd.DataFrame(all_food_urls, columns=['category', 'url'])
all_foods_df

,category,url
0,Món canh,https://www.dienmayxanh.com/vao-bep/2-cach-nau...
1,Món canh,https://www.dienmayxanh.com/vao-bep/cach-nau-g...
2,Món canh,https://www.dienmayxanh.com/vao-bep/3-cach-nau...
3,Món canh,https://www.dienmayxanh.com/vao-bep/cach-nau-s...
4,Món canh,https://www.dienmayxanh.com/vao-bep/cach-nau-s...
...,...,...
13510,Món cuốn - trộn,https://www.dienmayxanh.com/vao-bep/2-cach-lam...
13511,Món cuốn - trộn,https://www.dienmayxanh.com/vao-bep/cach-lam-c...
13512,Món cuốn - trộn,https://www.dienmayxanh.com/vao-bep/cach-lam-m...
13513,Món cuốn - trộn,https://www.dienmayxanh.com/vao-bep/cach-lam-c...


In [26]:
all_foods_df.to_csv('dienmayxanh_foods_urls.csv', index=False)
print("DataFrame successfully saved to output.csv")

DataFrame successfully saved to output.csv


## 3. Crawl chi tiết công thức

In [ ]:
def get_recipe_detail_dmx(url, category):
    result = {
        "link": url,
        "type_of_food": category,
        "title": None,
        "description": None,
        "author_name": None,
        "cook_time": None,
        "num_of_people": None,
        "calories": None,
        "num_of_ingredients": None,
        "ingredients": [],
        "step": [],
        "note": [],
        "post_date": None,
    }
    
    def safe_text(node):
        return node.get_text(strip=True) if node else None
    
    try:
        response = requests.get(url, headers=HEADERS, timeout=15)
        response.encoding = 'utf-8'
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Tìm vùng nội dung chính
        detail_content = soup.find('div', class_='detail-content')
        if not detail_content:
            detail_content = soup  # Fallback to toàn bộ page
        
        # Title - h1 đầu tiên trong detail-content
        try:
            title_tag = detail_content.find('h1')
            if title_tag:
                result['title'] = safe_text(title_tag)
        except:
            pass
        
        # Description - div.leadpost p
        try:
            leadpost = detail_content.find('div', class_='leadpost')
            if leadpost:
                result['description'] = safe_text(leadpost)
            else:
                # Fallback to meta description
                meta_desc = soup.find('meta', attrs={'name': 'description'})
                if meta_desc:
                    result['description'] = meta_desc.get('content', '')
        except:
            pass
        
        # Author - thử tìm nhiều vị trí
        try:
            # Từ script JSON-LD
            script_tag = soup.find('script', type='application/ld+json')
            if script_tag:
                import json
                try:
                    data = json.loads(script_tag.string)
                    if isinstance(data, dict) and 'author' in data:
                        author_data = data.get('author', {})
                        if isinstance(author_data, dict):
                            result['author_name'] = author_data.get('name')
                        elif isinstance(author_data, str):
                            result['author_name'] = author_data
                except:
                    pass
            
            if not result['author_name']:
                author_tag = soup.find('span', class_='author') or soup.find('a', class_='author')
                result['author_name'] = safe_text(author_tag)
        except:
            pass
        
        # Thông tin thời gian, số người từ ul.ready
        try:
            ready_items = soup.select('ul.ready li')
            for item in ready_items:
                h2_tag = item.find('h2')
                span_tag = item.find('span')
                if h2_tag and span_tag:
                    label = safe_text(h2_tag).lower()
                    value = safe_text(span_tag)
                    if 'chuẩn bị' in label or 'chế biến' in label:
                        if result['cook_time']:
                            result['cook_time'] += ' + ' + value
                        else:
                            result['cook_time'] = value
        except:
            pass
        
        # Số người - từ div.staple h2 small
        try:
            staple_div = soup.find('div', class_='staple')
            if staple_div:
                h2_tag = staple_div.find('h2')
                if h2_tag:
                    small_tag = h2_tag.find('small')
                    if small_tag:
                        result['num_of_people'] = safe_text(small_tag)
        except:
            pass
        
        # Ingredients - div.staple span
        try:
            ingredients = []
            staple_div = soup.find('div', class_='staple')
            if staple_div:
                ing_spans = staple_div.find_all('span', recursive=False)
                for span in ing_spans:
                    # Lấy tên nguyên liệu
                    name_parts = []
                    for content in span.contents:
                        if isinstance(content, str):
                            text = content.strip()
                            if text:
                                name_parts.append(text)
                    
                    name = ' '.join(name_parts).strip()
                    
                    # Lấy số lượng từ small
                    small_tag = span.find('small')
                    quantity = safe_text(small_tag) if small_tag else ''
                    
                    # Lấy ghi chú từ em
                    em_tag = span.find('em')
                    note = safe_text(em_tag) if em_tag else ''
                    
                    # Ghép lại
                    if name:
                        ingredient_text = name
                        if quantity:
                            ingredient_text += f" - {quantity}"
                        if note:
                            ingredient_text += f" ({note})"
                        ingredients.append(ingredient_text.strip())
            
            # Fallback: thử các selector khác
            if not ingredients:
                for selector in ['div.ingredient li', 'ul.ingredient li', 'div.nguyenlieu li']:
                    items = soup.select(selector)
                    if items:
                        for item in items:
                            text = safe_text(item)
                            if text and len(text) > 1:
                                ingredients.append(text)
                        break
            
            result['ingredients'] = ingredients
            result['num_of_ingredients'] = len(ingredients)
        except:
            pass
        
        # Steps - div.method ul li
        try:
            steps = []
            method_div = soup.find('div', class_='method')
            if method_div:
                step_items = method_div.find_all('li')
                for item in step_items:
                    # Lấy số bước từ label
                    label_tag = item.find('label')
                    step_num = safe_text(label_tag) if label_tag else ''
                    
                    # Lấy tiêu đề bước từ h3
                    h3_tag = item.find('h3')
                    step_title = safe_text(h3_tag) if h3_tag else ''
                    
                    # Lấy nội dung từ div.text-method
                    text_div = item.find('div', class_='text-method')
                    if text_div:
                        # Lấy tất cả p tags
                        p_tags = text_div.find_all('p')
                        contents = []
                        for p in p_tags:
                            text = safe_text(p)
                            if text:
                                contents.append(text)
                        step_content = ' '.join(contents)
                    else:
                        step_content = safe_text(item)
                    
                    # Ghép thành 1 bước
                    if step_content:
                        if step_num and step_title:
                            step_text = f"Bước {step_num} - {step_title}: {step_content}"
                        elif step_num:
                            step_text = f"Bước {step_num}: {step_content}"
                        else:
                            step_text = step_content
                        steps.append(step_text)
            
            # Fallback
            if not steps:
                for selector in ['ol.steps li', 'div.cachlam li', 'div.perform p']:
                    items = soup.select(selector)
                    if items:
                        for i, item in enumerate(items, 1):
                            text = safe_text(item)
                            if text and len(text) > 10:
                                text = re.sub(r'^(Bước\s*)?\d+[.:]?\s*', '', text)
                                steps.append(f"Bước {i}: {text}")
                        if steps:
                            break
            
            result['step'] = steps
        except:
            pass
        
        # Notes - div.tipsrecipe
        try:
            notes = []
            tips_divs = soup.select('div.tipsrecipe, div.infobox, div.tips')
            for tips in tips_divs:
                text = safe_text(tips)
                if text:
                    # Loại bỏ "Mách nhỏ:" prefix
                    text = re.sub(r'^Mách nhỏ:\s*', '', text)
                    notes.append(text)
            result['note'] = notes
        except:
            pass
        
        # Post date - thử tìm từ nhiều nguồn
        try:
            # Từ meta hoặc span.date
            date_tag = soup.find('span', class_='date') or soup.find('time')
            if date_tag:
                result['post_date'] = safe_text(date_tag)
        except:
            pass
        
        return result
        
    except Exception as e:
        print(f"  [ERROR] {url}: {e}")
        return result

In [9]:
# Test
if len(all_recipe_urls) > 0:
    test_cat, test_url = all_recipe_urls[0]
    print(f"Testing: {test_url}")
    detail = get_recipe_detail_dmx(test_url, test_cat)
    print(f"\nTitle: {detail['title']}")
    print(f"Description: {detail['description'][:100] if detail['description'] else 'N/A'}...")
    print(f"Ingredients ({len(detail['ingredients'])}): {detail['ingredients'][:3]}")
    print(f"Steps ({len(detail['step'])}): {detail['step'][:1] if detail['step'] else 'N/A'}")

Testing: https://www.dienmayxanh.com/vao-bep/cach-lam-goi-du-du-tom-kho-hap-dan-don-gian-14052

Title: Cách làm gỏi đu đủ tôm khô hấp dẫn, đơn giản
Description: Nếu bạn đang tìm mộtmón gỏivừa có thể ăn vặt vừa có thể nhâm nhi vào dịp cuối tuần nhưng lại vô cùng...
Ingredients (7): ['Đu đủ xanh - 1 trái', 'Tôm khô - 100 gr', 'Chanh - 1 trái']
Steps (6): ['Bước 1 - Sơ chế nguyên liệu: Đu đủ mua về bạn cắt đôi, bỏ phần hạt bên trong đu đủ rồi dùng dụng cụ bào sợi, bào bên trong của trái đu đủ cho đến khi hết phần thịt. Tiếp theo, bạn cho đu đủ đã bào vào thau nước lạnh, nhồi rửa đu đủ rồi vớt ra rổ để ráo nước. Rau răm mua về bạn nhặt lấy lá, sau đó rửa sạch lại với nước rồi vớt ra để ráo. Nếu thích ăn rau răm nhỏ thì bạn có thể cắt làm đôi hoặc ba đều được nhé!']

Title: Cách làm gỏi đu đủ tôm khô hấp dẫn, đơn giản
Description: Nếu bạn đang tìm mộtmón gỏivừa có thể ăn vặt vừa có thể nhâm nhi vào dịp cuối tuần nhưng lại vô cùng...
Ingredients (7): ['Đu đủ xanh - 1 trái', 'Tôm khô - 100 gr

In [10]:
# Load URLs từ file
if os.path.exists('dmx_recipe_urls.csv'):
    urls_df = pd.read_csv('dmx_recipe_urls.csv')
    all_recipe_urls = list(zip(urls_df['category'], urls_df['url']))
    print(f"Loaded {len(all_recipe_urls)} URLs")

Loaded 269 URLs


In [11]:
# Crawl chi tiết
all_recipes = []
failed_urls = []
CHECKPOINT = 100

for i, (category, url) in enumerate(tqdm(all_recipe_urls, desc="Crawling")):
    try:
        detail = get_recipe_detail_dmx(url, category)
        
        if detail['title'] and len(detail['ingredients']) > 0:
            all_recipes.append(detail)
        else:
            failed_urls.append((category, url, "Missing data"))
        
        if (i + 1) % CHECKPOINT == 0:
            pd.DataFrame(all_recipes).to_csv('dmx_recipes_checkpoint.csv', index=False)
            print(f"\n  💾 Checkpoint: {len(all_recipes)} recipes")
        
        time.sleep(random.uniform(MIN_DELAY, MAX_DELAY))
        
    except Exception as e:
        failed_urls.append((category, url, str(e)))

print(f"\n✅ Success: {len(all_recipes)}")
print(f"❌ Failed: {len(failed_urls)}")

Crawling:  37%|███▋      | 99/269 [02:43<04:29,  1.59s/it]


  💾 Checkpoint: 90 recipes


Crawling:  74%|███████▍  | 199/269 [05:28<01:44,  1.49s/it]


  💾 Checkpoint: 186 recipes


Crawling: 100%|██████████| 269/269 [07:16<00:00,  1.62s/it]


✅ Success: 252
❌ Failed: 17


In [12]:
# Lưu kết quả
dmx_df = pd.DataFrame(all_recipes)
dmx_df.to_csv('dmx_recipes_detail.csv', index=False)
print(f"Saved {len(dmx_df)} recipes")

if failed_urls:
    pd.DataFrame(failed_urls, columns=['category', 'url', 'error']).to_csv('dmx_failed_urls.csv', index=False)

Saved 252 recipes


## 4. Merge tất cả nguồn dữ liệu

In [13]:
# Load tất cả sources
dataframes = []

# VnExpress
if os.path.exists('vnexpress_foods_detail_merged.csv'):
    vne_df = pd.read_csv('vnexpress_foods_detail_merged.csv')
    vne_df['source'] = 'vnexpress.net'
    dataframes.append(vne_df)
    print(f"VnExpress: {len(vne_df)} recipes")

# Cooky
if os.path.exists('cooky_recipes_detail.csv'):
    cooky_df = pd.read_csv('cooky_recipes_detail.csv')
    dataframes.append(cooky_df)
    print(f"Cooky: {len(cooky_df)} recipes")

# DMX
if os.path.exists('dmx_recipes_detail.csv'):
    dmx_df = pd.read_csv('dmx_recipes_detail.csv')
    dataframes.append(dmx_df)
    print(f"DMX: {len(dmx_df)} recipes")

VnExpress: 893 recipes
DMX: 252 recipes


In [14]:
# Chuẩn hóa cột
common_cols = [
    'link', 'type_of_food', 'title', 'description', 'author_name',
    'cook_time', 'num_of_people', 'calories', 'num_of_ingredients',
    'ingredients', 'step', 'note', 'post_date', 'source'
]

for i, df in enumerate(dataframes):
    for col in common_cols:
        if col not in df.columns:
            df[col] = None
    dataframes[i] = df[common_cols]

In [15]:
# Merge
if dataframes:
    merged_df = pd.concat(dataframes, ignore_index=True)
    merged_df = merged_df.drop_duplicates(subset=['title'], keep='first')
    
    print(f"\n📊 Final Merged Dataset:")
    print(f"  Total: {len(merged_df)} recipes")
    print(f"\n  By source:")
    print(merged_df['source'].value_counts())
    
    # Lưu
    merged_df.to_csv('all_recipes_final.csv', index=False)
    print(f"\n✅ Saved to all_recipes_final.csv")


📊 Final Merged Dataset:
  Total: 1138 recipes

  By source:
source
vnexpress.net      886
dienmayxanh.com    252
Name: count, dtype: int64

✅ Saved to all_recipes_final.csv

✅ Saved to all_recipes_final.csv


In [16]:
# Thống kê cuối cùng
print("📈 Thống kê loại món:")
print(merged_df['type_of_food'].value_counts().head(20))

📈 Thống kê loại món:
type_of_food
Món ngon hàng ngày             486
Món ngon cho cuối tuần         123
Món Tết                         81
Món tráng miệng, giải khát      56
Quà - Món ăn vặt                51
Món ngon theo vùng miền         24
Món Chay                        24
Món Chè                         23
Món Nướng                       22
Món Hấp                         22
Món Gỏi                         21
Món ngon ngày lạnh              21
Món Kho                         20
Món Xào                         20
Món Chiên                       20
Món Lẩu                         19
Thực đơn cho ngày nắng nóng     19
Món Bánh                        18
Món Thịt Heo                    16
Món Canh                        15
Name: count, dtype: int64
